In [ ]:
import base64, json, os, shutil, subprocess, sys
from pathlib import Path

JOB_MANIFEST = json.loads(base64.b64decode("__JOB_MANIFEST_B64__"))
print({k: (v if k != "prompt" else v[:180]) for k, v in JOB_MANIFEST.items()})
assert int(JOB_MANIFEST["frame_num"]) >= 5 and (int(JOB_MANIFEST["frame_num"]) - 1) % 4 == 0
assert int(JOB_MANIFEST["sample_steps"]) >= 1


In [ ]:
WORK = Path("/kaggle/working")
WAN_DIR = WORK / "Wan2.1"
MODEL_DIR = WORK / "Wan2.1-T2V-1.3B"
OUT_DIR = WORK / "wan_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Wan-Video/Wan2.1.git", str(WAN_DIR)], check=True)
req = WORK / "requirements_no_flash.txt"
lines = (WAN_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines()
req.write_text("\n".join(line for line in lines if "flash_attn" not in line.lower() and "flash-attn" not in line.lower()) + "\n", encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord", "ftfy", "dashscope", "huggingface_hub"], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is not enabled for this kernel.")
print("GPU:", torch.cuda.get_device_name(0), "CUDA:", torch.version.cuda)


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="Wan-AI/Wan2.1-T2V-1.3B", local_dir=str(MODEL_DIR))
model_py = WAN_DIR / "wan/modules/model.py"
source = model_py.read_text(encoding="utf-8")
source = source.replace("from .attention import flash_attention", "from .attention import attention")
source = source.replace("flash_attention(", "attention(")
if "from .attention import attention" not in source or "flash_attention(" in source:
    raise RuntimeError("Wan T4 SDPA fallback patch verification failed.")
model_py.write_text(source, encoding="utf-8")
print("Wan T4 SDPA fallback patch applied.")


In [ ]:
prompt = str(JOB_MANIFEST["prompt"]).strip()
if not prompt:
    raise ValueError("Prompt must not be empty.")
output_path = OUT_DIR / "wan_scene_001.mp4"
command = [
    sys.executable, "generate.py",
    "--task", "t2v-1.3B",
    "--size", str(JOB_MANIFEST["size"]),
    "--frame_num", str(JOB_MANIFEST["frame_num"]),
    "--sample_steps", str(JOB_MANIFEST["sample_steps"]),
    "--ckpt_dir", str(MODEL_DIR),
    "--offload_model", "True",
    "--t5_cpu",
    "--sample_shift", "8",
    "--sample_guide_scale", "5",
    "--save_file", str(output_path),
    "--prompt", prompt,
]
print("Starting Wan2.1 generation...")
subprocess.run(command, cwd=str(WAN_DIR), check=True)
if not output_path.is_file() or output_path.stat().st_size < 50_000:
    raise RuntimeError("Wan2.1 output missing or too small.")
probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_type,width,height", "-of", "json", str(output_path)],
    capture_output=True, text=True, check=True
)
probe_data = json.loads(probe.stdout)
video_stream = next((s for s in probe_data.get("streams", []) if s.get("codec_type") == "video"), None)
duration = float((probe_data.get("format") or {}).get("duration") or 0)
if not video_stream or duration <= 0:
    raise RuntimeError("FFprobe could not validate generated video.")
print("SUCCESS:", output_path, "size_MB=", round(output_path.stat().st_size / 1024 / 1024, 2), "duration_s=", round(duration, 2), "resolution=", video_stream.get("width"), "x", video_stream.get("height"))


In [ ]:
print("Worker outputs:")
for item in sorted(OUT_DIR.glob("*.mp4")):
    print(item.name, item.stat().st_size)
assert list(OUT_DIR.glob("*.mp4")), "No MP4 was generated."


In [ ]:
# Keep Kaggle's published kernel output small: retain only the deliverables.
# Wan2.1 source and model weights are temporary inference dependencies, not artifacts.
for temporary_path in (WAN_DIR, MODEL_DIR):
    if temporary_path.exists():
        print(f"Removing temporary inference directory: {temporary_path}")
        shutil.rmtree(temporary_path, ignore_errors=True)
req.unlink(missing_ok=True)
deliverables = sorted(OUT_DIR.glob("*.mp4"))
if not deliverables:
    raise RuntimeError("No generated MP4 remains in the deliverables directory.")
print("Final Kaggle deliverables:")
for item in deliverables:
    print(item.name, item.stat().st_size)
print("Temporary model and source directories removed.")
